In [ ]:
%pip install pandas

KPI 1 : Score par ville

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data_base_copro.csv')
df.columns = df.columns.str.strip().str.lower()

cols_numeriques = ['lots_habitation', 'lots_parking', 'total_lots']
for col in cols_numeriques:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

if 'arrondissement' in df.columns:
    df['ville'] = df['arrondissement'].fillna(df['ville'])

df['ville'] = df['ville'].astype(str).str.upper()
df['ville'] = df['ville'].str.replace('ČME', 'ÈME', regex=False)

df_propre = df[df['lots_habitation'] >= 5].copy()
df_propre['ratio_habitation'] = df_propre['lots_habitation'] / df_propre['total_lots'].replace(0, np.nan)
df_propre = df_propre[df_propre['ratio_habitation'] >= 0.3].copy()

df_unique = df_propre.groupby(['adresse', 'ville', 'code_postal', 'dept_code', 'dept_nom']).agg({
    'lots_habitation': 'max',
    'lots_parking': 'max',
    'lat': 'first',
    'long': 'first'
}).reset_index()

df_unique['score_immeuble'] = df_unique['lots_parking'] / df_unique['lots_habitation'].replace(0, 1)

df_final = df_unique[(df_unique['score_immeuble'] >= 0.5) & (df_unique['score_immeuble'] <= 0.9)].copy()
df_final['code_postal'] = df_final['code_postal'].fillna(0).astype(int).astype(str)

kpi1_score_ville = df_final.groupby(['code_postal', 'ville']).agg(
    nb_immeubles_cibles=('adresse', 'count'),
    total_lots_cibles=('lots_habitation', 'sum'),
    score_moyen_cibles=('score_immeuble', 'mean')
).reset_index()

kpi1_score_ville['score_moyen_cibles'] = kpi1_score_ville['score_moyen_cibles'].round(2)

kpi1_score_ville = kpi1_score_ville.sort_values('total_lots_cibles', ascending=False).reset_index(drop=True)

display(kpi1_score_ville.head(10))

,code_postal,ville,nb_immeubles_cibles,total_lots_cibles,score_moyen_cibles
0,75016,PARIS 16ÈME ARRONDISSEMENT,247,12519,0.67
1,75015,PARIS 15ÈME ARRONDISSEMENT,208,11274,0.71
2,21000,DIJON,273,9874,0.68
3,92100,BOULOGNE-BILLANCOURT,185,9663,0.71
4,34300,AGDE,106,8999,0.72
5,78150,LE CHESNAY-ROCQUENCOURT,16,8298,0.66
6,69100,VILLEURBANNE,179,8140,0.73
7,75020,PARIS 20ÈME ARRONDISSEMENT,130,7930,0.73
8,44000,NANTES,269,7850,0.71
9,63000,CLERMONT-FERRAND,237,7568,0.70


Export des données pour l'équipe Front-End :

In [ ]:
kpi1_score_ville.to_csv('export_kpi1_villes.csv', index=False, encoding='utf-8')